# Vignette gallery, end-to-end tour

Reproduces the robust-estimator core of `fitmodelsRobStatTM.R` and `VignetteRobStatTM.R` (the `fit.models` framework itself is out of scope).

In [ ]:
import os, sys, pathlib


import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless-safe for CI execution
import matplotlib.pyplot as plt
import robstattm_py as rpm
from robstattm_py import set_seed
from robstattm_py._r import r as _r

ro = _r()
ro.r("suppressMessages(library(RobStatTM))")
FIG_DIR = pathlib.Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print(f"robstattm_py {rpm.__version__}")

## End-to-end vignette (`fitmodelsRobStatTM.R`, `VignetteRobStatTM.R`)

These scripts walk through RobStatTM via the **fit.models** comparison framework (`fit.models`, `plot.lmfm`, `plot.covfm`). `fit.models` is a separate package outside this project's scope, so here we reproduce the **robust estimator core** the vignette is built on, `lmrobdetMM` with the `mopt`/0.95 default control, and `covClassic` vs `covRob` on wine, and compare them using our own dataclass summaries instead of `fit.models`.

### Regression: LS vs `lmrobdetMM` on mineral (mopt, eff 0.95)

In [ ]:
mineral = rpm.datasets.mineral()
control = rpm.lmrobdet_control(family='mopt', efficiency=0.95)
robfit = rpm.lmrobdet_mm('zinc ~ copper', mineral, control=control)
import numpy as np
Xm = np.c_[np.ones(len(mineral)), mineral['copper'].to_numpy(float)]
ym = mineral['zinc'].to_numpy(float)
ls = np.linalg.lstsq(Xm, ym, rcond=None)[0]
print('LS    coef:', np.round(ls, 3))
print('robust coef:', np.round(robfit.coefficients, 3))
print('\nrobust summary:')
print(robfit.summary().coefficients_table)

Strict-tier cross-check vs direct R `lmrobdetMM`:

In [ ]:
# lmrobdetMM is deterministic (Peña–Yohai initial), so no seeding is needed.
robfit_chk = rpm.lmrobdet_mm('zinc ~ copper', mineral, control=control)
ro.r('data(mineral); ctrl <- lmrobdet.control(family="mopt", efficiency=0.95)')
ro.r('rfit <- lmrobdetMM(zinc ~ copper, control=ctrl, data=mineral)')
r_coef = np.asarray(ro.r('as.numeric(rfit$coefficients)'), dtype=float)
print('coefficients bit-equal to R:', np.array_equal(robfit_chk.coefficients, r_coef))

### Covariance: `covClassic` vs `covRob` on wine[, 1:5]

In [ ]:
wine5 = rpm.datasets.wine().iloc[:, :5]
cl = rpm.cov_classic(wine5)
set_seed(1)
rb = rpm.cov_rob(wine5, type='auto')
print('classic eigenvalues:', np.round(cl.summary().evals, 3))
print('robust  eigenvalues:', np.round(rb.summary().evals, 3))
print('robust estimator chosen by covRob(type="auto"):', rb.estimator_type)

The robust eigen-spectrum differs from the classical one because a few high-leverage wines inflate the classical covariance, exactly the comparison the vignette's `fit.models` plots illustrate. Every number above matches the underlying R call bit-for-bit.